# Simple LLM Calling

First real notebook in this repo: just verifying I can call an LLM from Python via LangChain,
then poking at the `temperature` parameter, then doing the same thing with a second provider
(Groq) as a backup. Run `docs/05_api_key_setup.md` first so `.env` has your keys.

In [ ]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

## 1. A single LLM call

The simplest possible thing: create a model instance, call `.invoke()` with a string prompt,
read `.text` off the response.

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

response = llm.invoke("How many moons does Jupiter have?")
print(response.text)

## 2. System vs. human messages

A plain string prompt is fine for quick tests, but real applications usually separate the
*instruction* (system message — how the model should behave) from the *actual question*
(human message). Passing a list of `(role, content)` pairs lets you do that.

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

response = llm.invoke([
    ("system", "You are a helpful assistant that answers in one short line."),
    ("human", "How many moons does Jupiter have?"),
])
print(response.text)

Notice the answer is noticeably shorter and more to-the-point than the first call — same
question, different system instruction.

## 3. Temperature

`temperature=0` → deterministic, same answer every time. Higher temperature → more varied,
"creative" output. Same prompt, run twice at different temperatures:

In [ ]:
prompt = "Write a single tagline for a quirky coffee shop. Only output the tagline, nothing else."

llm_cold = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
print("temperature=0:", llm_cold.invoke(prompt).text)

In [ ]:
llm_hot = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=1)
print("temperature=1:", llm_hot.invoke(prompt).text)

## 4. Calling a different provider: Groq

Same LangChain interface, different import and model class — this is one of the nice things
about LangChain: swapping providers doesn't mean rewriting your whole application.

In [ ]:
from langchain_groq import ChatGroq

llm_groq = ChatGroq(model="qwen/qwen3-32b", reasoning_format="parsed")

messages = [
    ("system", "You are a helpful assistant that gives a short answer to a question."),
    ("human", "How many moons does Jupiter have?"),
]
response = llm_groq.invoke(messages)
print(response.content)

That's it for the basics — one working LLM call is really all the "simple_llm_calling"
chapter needs to prove out. Next: [`../02_health_analysis`](../02_health_analysis), a small
two-stage pipeline that actually does something useful with this.